# DeBERTa Nested CV — Context L4 — batch32 FP32 A100 trial, LR grid [2e-5, 5e-5, 8e-5]


This trial keeps the dataset, fold structure, metrics, and learning-rate tuning intact. It only changes training-side engineering settings: `batch_size=32`, FP32, `LR_VALUES=[2e-5, 5e-5, 8e-5]`, and a separate `RUN_TAG` so results do not mix with earlier runs.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

!pip install -q -U transformers accelerate sentencepiece



Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 135.7 MB/s eta 0:00:00


In [ ]:

import os
import gc
import re
import math
import json
import random
import unicodedata
import subprocess
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)


TOP_K_CATEGORIES = 10
SAMPLE_PER_CLASS = 2000
RANDOM_STATE = 42


CONTEXT_COLUMN = "L4"

OUTER_FOLDS = 10
INNER_FOLDS = 3


LR_VALUES = [2e-5, 5e-5, 8e-5]

DEBERTA_MODEL = "microsoft/deberta-base"
DEBERTA_MAX_LENGTH = 96


DEBERTA_BATCH_SIZE = 32
DEBERTA_NUM_EPOCHS = 3
DEBERTA_WEIGHT_DECAY = 0.01
DEBERTA_WARMUP_RATIO = 0.10

USE_MIXED_PRECISION = False
USE_BF16 = False
USE_FP16 = False

DEBUG_TRAINING = True                  # prints epoch-level mean/last loss
PRINT_PRED_DISTRIBUTION = True         # detects one-class prediction collapse
STOP_ON_SINGLE_CLASS_PREDICTION = True # prevents saving invalid collapsed folds

REQUIRE_A100_FOR_BATCH32 = False

RUN_TAG = "stable_deberta_base_bs32_fp32_lr2e5_5e5_8e5"

# ----- I/O -----
DATA_JSON_PATH = "/content/drive/MyDrive/Colab Notebooks/SML/News_Category_Dataset_v3.json"
SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/SML"
PROGRESS_PATH = f"{SAVE_DIR}/deberta_{CONTEXT_COLUMN}_{RUN_TAG}_fold_progress.csv"
PER_CLASS_PATH = f"{SAVE_DIR}/deberta_{CONTEXT_COLUMN}_{RUN_TAG}_per_class_f1.csv"
SUMMARY_PATH = f"{SAVE_DIR}/deberta_{CONTEXT_COLUMN}_{RUN_TAG}_summary.csv"

os.makedirs(SAVE_DIR, exist_ok=True)


print("===== nvidia-smi =====")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print("Could not run nvidia-smi:", repr(exc))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0)
    print("GPU:", GPU_NAME)
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    if DEBERTA_BATCH_SIZE >= 32 and "A100" not in GPU_NAME:
        message = (
            f"WARNING: DEBERTA_BATCH_SIZE={DEBERTA_BATCH_SIZE}, but GPU is '{GPU_NAME}', not A100. "
            "For T4/P100, batch size 32 may OOM or slow down. Set DEBERTA_BATCH_SIZE=16 if this happens."
        )
        print(message)
        if REQUIRE_A100_FOR_BATCH32:
            raise RuntimeError(message)
else:
    print("WARNING: no GPU detected. Switch Runtime -> Change runtime type -> GPU.")

assert USE_MIXED_PRECISION is False
assert USE_BF16 is False
assert USE_FP16 is False
assert isinstance(DEBERTA_BATCH_SIZE, int) and DEBERTA_BATCH_SIZE > 0

print("DEBERTA_MODEL:", DEBERTA_MODEL)
print("DEBERTA_BATCH_SIZE:", DEBERTA_BATCH_SIZE)
print("DEBERTA_NUM_EPOCHS:", DEBERTA_NUM_EPOCHS)
print("LR_VALUES:", LR_VALUES)
print("USE_MIXED_PRECISION:", USE_MIXED_PRECISION)
print("USE_BF16:", USE_BF16)
print("USE_FP16:", USE_FP16)
print("Context level for this notebook:", CONTEXT_COLUMN)
print("Progress path:", PROGRESS_PATH)


===== nvidia-smi =====
Wed May 20 06:23:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------------------

## Data preprocessing 

In [ ]:

data = pd.read_json(DATA_JSON_PATH, lines=True)

data = data[["category", "headline", "short_description"]]
data = data.dropna()

data["category"] = data["category"].astype(str).str.strip()
data["headline"] = data["headline"].astype(str).str.strip()
data["short_description"] = data["short_description"].astype(str).str.strip()

data = data[
    (data["category"] != "") &
    (data["headline"] != "") &
    (data["short_description"] != "")
].copy()

top_10_categories = data["category"].value_counts().head(TOP_K_CATEGORIES).index.tolist()
data = data[data["category"].isin(top_10_categories)].copy()

sampled_data = []
for category in top_10_categories:
    category_data = data[data["category"] == category]
    category_sample = category_data.sample(n=SAMPLE_PER_CLASS, random_state=RANDOM_STATE)
    sampled_data.append(category_sample)

data = pd.concat(sampled_data)
data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

category_to_label = {}
for i in range(len(top_10_categories)):
    category_to_label[top_10_categories[i]] = i

labels = []
for category in data["category"]:
    labels.append(category_to_label[category])
data["label"] = labels

def get_first_tokens(text, number_of_tokens):
    tokens = str(text).split()
    return " ".join(tokens[:number_of_tokens])

data["L1"] = data["headline"].apply(lambda x: get_first_tokens(x, 5))
data["L2"] = data["headline"]
data["L3"] = data["headline"] + " " + data["short_description"].apply(
    lambda x: get_first_tokens(x, 15)
)
data["L4"] = data["headline"] + " " + data["short_description"]

data = data[
    ["category", "label", "headline", "short_description", "L1", "L2", "L3", "L4"]
].copy()

print("Data shape:", data.shape)
print("Categories per label:")
print(data["category"].value_counts())

# Materialize the arrays this notebook actually uses
X_text_all = data[CONTEXT_COLUMN].astype(str).values
y_all = data["label"].values
print(f"\nUsing context column: {CONTEXT_COLUMN}")
print(f"X_text_all shape: {X_text_all.shape}, y_all shape: {y_all.shape}")


Data shape: (20000, 8)
Categories per label:
category
PARENTING         2000
WELLNESS          2000
TRAVEL            2000
POLITICS          2000
FOOD & DRINK      2000
BUSINESS          2000
STYLE & BEAUTY    2000
HEALTHY LIVING    2000
ENTERTAINMENT     2000
QUEER VOICES      2000
Name: count, dtype: int64

Using context column: L4
X_text_all shape: (20000,), y_all shape: (20000,)


## From-scratch CV folds + metrics

In [ ]:
def make_stratified_folds(y, number_of_folds, random_state):
    y = np.array(y)
    rng = np.random.default_rng(random_state)

    folds = []
    for i in range(number_of_folds):
        folds.append([])

    unique_labels = np.unique(y)
    for label in unique_labels:
        label_indices = np.where(y == label)[0]
        rng.shuffle(label_indices)
        split_indices = np.array_split(label_indices, number_of_folds)
        for fold_number in range(number_of_folds):
            folds[fold_number].extend(split_indices[fold_number].tolist())

    final_folds = []
    for fold in folds:
        fold = np.array(fold)
        rng.shuffle(fold)
        final_folds.append(fold)
    return final_folds


def calculate_accuracy(y_true, y_pred):
    correct_count = 0
    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            correct_count += 1
    return correct_count / len(y_true)


def calculate_macro_f1(y_true, y_pred):
    labels = np.unique(y_true)
    f1_scores = []
    for label in labels:
        tp = fp = fn = 0
        for i in range(len(y_true)):
            if y_true[i] == label and y_pred[i] == label:
                tp += 1
            elif y_true[i] != label and y_pred[i] == label:
                fp += 1
            elif y_true[i] == label and y_pred[i] != label:
                fn += 1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
    return float(np.mean(f1_scores))


def calculate_weighted_f1(y_true, y_pred):
    labels = np.unique(y_true)
    total_count = len(y_true)
    weighted_sum = 0.0
    for label in labels:
        tp = fp = fn = support = 0
        for i in range(len(y_true)):
            if y_true[i] == label:
                support += 1
            if y_true[i] == label and y_pred[i] == label:
                tp += 1
            elif y_true[i] != label and y_pred[i] == label:
                fp += 1
            elif y_true[i] == label and y_pred[i] != label:
                fn += 1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        weighted_sum += f1 * support
    return weighted_sum / total_count


def calculate_per_class_f1(y_true, y_pred):
    labels = np.unique(y_true)
    result = {}
    for label in labels:
        tp = fp = fn = 0
        for i in range(len(y_true)):
            if y_true[i] == label and y_pred[i] == label:
                tp += 1
            elif y_true[i] != label and y_pred[i] == label:
                fp += 1
            elif y_true[i] == label and y_pred[i] != label:
                fn += 1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        result[int(label)] = f1
    return result


## DeBERTa fine-tune helper




In [ ]:


class TextClassificationDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


def _set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _prediction_distribution(y_pred):
    unique, counts = np.unique(y_pred, return_counts=True)
    return {int(k): int(v) for k, v in zip(unique, counts)}


def fine_tune_deberta_and_predict(
    X_train_text,
    y_train,
    X_eval_text,
    learning_rate,
    num_labels,
    epochs=None,
    batch_size=None,
    max_length=None,
    use_bf16=None,
    use_fp16=None,
    seed=42,
    run_name="",
):
    if epochs is None:
        epochs = DEBERTA_NUM_EPOCHS
    if batch_size is None:
        batch_size = DEBERTA_BATCH_SIZE
    if max_length is None:
        max_length = DEBERTA_MAX_LENGTH
    if use_bf16 is None:
        use_bf16 = USE_BF16
    if use_fp16 is None:
        use_fp16 = USE_FP16

    if isinstance(batch_size, bool):
        raise ValueError(f"Invalid batch_size={batch_size}. Expected a positive integer, e.g. 16 or 32.")
    batch_size = int(batch_size)
    epochs = int(epochs)
    max_length = int(max_length)

    if batch_size <= 0:
        raise ValueError(f"Invalid batch_size={batch_size}. Expected a positive integer.")
    if epochs <= 0:
        raise ValueError(f"Invalid epochs={epochs}. Expected a positive integer.")
    if max_length <= 0:
        raise ValueError(f"Invalid max_length={max_length}. Expected a positive integer.")

    y_train = np.asarray(y_train, dtype=np.int64)
    label_min = int(np.min(y_train))
    label_max = int(np.max(y_train))
    if label_min < 0 or label_max >= num_labels:
        raise ValueError(
            f"Label range [{label_min}, {label_max}] is invalid for num_labels={num_labels}."
        )

    assert not use_bf16 and not use_fp16, "This trial notebook is intended to run FP32 only."

    _set_all_seeds(seed)

    tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL, use_fast=True)

    train_enc = tokenizer(
        list(X_train_text),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    eval_enc = tokenizer(
        list(X_eval_text),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    train_dataset = TextClassificationDataset(train_enc, y_train)
    eval_dataset = TextClassificationDataset(eval_enc, np.zeros(len(X_eval_text)))

    train_generator = torch.Generator()
    train_generator.manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=train_generator,
    )
    eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        DEBERTA_MODEL,
        num_labels=num_labels,
        problem_type="single_label_classification",
        use_safetensors=False,
    )
    model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=DEBERTA_WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * DEBERTA_WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    amp_enabled = bool(device.type == "cuda" and (use_bf16 or use_fp16))
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=bool(device.type == "cuda" and use_fp16))

    if DEBUG_TRAINING:
        print(
            f"      Train call {run_name} | n_train={len(y_train)} n_eval={len(X_eval_text)} "
            f"lr={learning_rate:.0e} epochs={epochs} batch={batch_size} "
            f"bf16={use_bf16} fp16={use_fp16}"
        )

    # ----- Training loop -----
    model.train()
    for epoch in range(epochs):
        epoch_losses = []
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            if amp_enabled:
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    outputs = model(**batch)
                    loss = outputs.loss
            else:
                outputs = model(**batch)
                loss = outputs.loss

            if torch.isnan(loss).item():
                raise RuntimeError(f"NaN loss detected in {run_name}. Stop this run. This notebook is FP32; reduce LR_VALUES or check labels/input text.")

            if use_fp16 and device.type == "cuda":
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            scheduler.step()
            epoch_losses.append(float(loss.detach().cpu().item()))

        if DEBUG_TRAINING:
            print(
                f"      epoch={epoch + 1}/{epochs} "
                f"mean_loss={np.mean(epoch_losses):.4f} "
                f"last_loss={epoch_losses[-1]:.4f}"
            )

    # ----- Prediction -----
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in eval_loader:
            forward_kwargs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }
            if "token_type_ids" in batch:
                forward_kwargs["token_type_ids"] = batch["token_type_ids"].to(device)

            if amp_enabled:
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    outputs = model(**forward_kwargs)
            else:
                outputs = model(**forward_kwargs)

            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.append(preds)

    preds_all = np.concatenate(all_preds)
    pred_dist = _prediction_distribution(preds_all)
    if PRINT_PRED_DISTRIBUTION:
        print(f"      Prediction distribution {run_name}: {pred_dist}")

    if STOP_ON_SINGLE_CLASS_PREDICTION and len(pred_dist) == 1:
        raise RuntimeError(
            f"Prediction collapsed to a single class in {run_name}: {pred_dist}. "
            "This usually indicates failed fine-tuning, unstable mixed precision, "
            "or an overly aggressive batch/learning-rate setting. No fold result was saved."
        )

    del model, optimizer, scheduler, train_loader, eval_loader
    del train_dataset, eval_dataset, train_enc, eval_enc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return preds_all

## Inner CV for learning-rate selection 

In [ ]:


def tune_lr_with_inner_cv(X_outer_train_text, y_outer_train, lr_values,
                          inner_folds_number, random_state, num_labels,
                          context_label=""):
    """
    For each candidate learning rate, run 3-fold inner CV on the outer-train set.
    Return the lr with the highest average inner macro-F1.
    """
    inner_folds = make_stratified_folds(y_outer_train, inner_folds_number, random_state)

    lr_to_score = {}
    for lr in lr_values:
        inner_f1s = []
        for inner_fold_index in range(inner_folds_number):
            valid_indices = inner_folds[inner_fold_index]
            all_indices = np.arange(len(y_outer_train))
            train_indices = np.setdiff1d(all_indices, valid_indices)

            X_inner_train = X_outer_train_text[train_indices]
            y_inner_train = y_outer_train[train_indices]
            X_inner_valid = X_outer_train_text[valid_indices]
            y_inner_valid = y_outer_train[valid_indices]

            run_name = f"{context_label} inner_lr={lr:.0e}_fold={inner_fold_index}"
            y_pred = fine_tune_deberta_and_predict(
                X_inner_train,
                y_inner_train,
                X_inner_valid,
                learning_rate=lr,
                num_labels=num_labels,
                seed=RANDOM_STATE + 1000 * int(random_state) + 100 * inner_fold_index + int(round(lr * 1e6)),
                run_name=run_name,
            )
            macro_f1 = calculate_macro_f1(y_inner_valid, y_pred)
            inner_f1s.append(macro_f1)
            print(f"    [Inner] {context_label} lr={lr:.0e}  fold={inner_fold_index}  macroF1={macro_f1:.4f}")

        avg_f1 = float(np.mean(inner_f1s))
        lr_to_score[lr] = avg_f1
        print(f"  [Inner] {context_label} lr={lr:.0e}  avg macroF1={avg_f1:.4f}")

    best_lr = max(lr_to_score, key=lr_to_score.get)
    return best_lr, lr_to_score[best_lr], lr_to_score

## Main nested CV loop 

In [ ]:

outer_folds = make_stratified_folds(y_all, OUTER_FOLDS, RANDOM_STATE)
num_labels = int(len(np.unique(y_all)))
print(f"Outer fold count: {len(outer_folds)}")
print(f"Number of classes: {num_labels}")
print("Progress path:", PROGRESS_PATH)

if os.path.exists(PROGRESS_PATH):
    progress_df = pd.read_csv(PROGRESS_PATH)
    progress_df = progress_df.drop_duplicates(subset=["outer_fold"], keep="last")
    completed_folds = set(progress_df["outer_fold"].astype(int).tolist())
    print(f"\nResume mode: {len(completed_folds)} folds already done -> {sorted(completed_folds)}")
else:
    completed_folds = set()
    print("\nFresh start. No prior progress file found.")

# ----- Main loop -----
for outer_fold_index in range(OUTER_FOLDS):
    if outer_fold_index in completed_folds:
        print(f"\n>> Outer fold {outer_fold_index}: already done, skipping.")
        continue

    test_indices = outer_folds[outer_fold_index]
    train_indices = np.setdiff1d(np.arange(len(y_all)), test_indices)

    X_outer_train_text = X_text_all[train_indices]
    y_outer_train = y_all[train_indices]
    X_outer_test_text = X_text_all[test_indices]
    y_outer_test = y_all[test_indices]

    print(f"\n{'='*60}")
    print(f">> Context {CONTEXT_COLUMN} | Outer fold {outer_fold_index} | train={len(y_outer_train)} test={len(y_outer_test)}")
    print(f"{'='*60}")

    # ----- Inner CV: pick best lr (full nested CV) -----
    best_lr, best_inner_macro_f1, all_lr_scores = tune_lr_with_inner_cv(
        X_outer_train_text=X_outer_train_text,
        y_outer_train=y_outer_train,
        lr_values=LR_VALUES,
        inner_folds_number=INNER_FOLDS,
        random_state=outer_fold_index,
        num_labels=num_labels,
        context_label=f"{CONTEXT_COLUMN}/outer{outer_fold_index}",
    )
    print(f">> Best lr for outer fold {outer_fold_index}: {best_lr:.0e}  (inner macroF1={best_inner_macro_f1:.4f})")

    # ----- Outer evaluation: retrain on full outer-train with best lr -----
    y_test_pred = fine_tune_deberta_and_predict(
        X_outer_train_text,
        y_outer_train,
        X_outer_test_text,
        learning_rate=best_lr,
        num_labels=num_labels,
        seed=RANDOM_STATE + 10000 + outer_fold_index,
        run_name=f"{CONTEXT_COLUMN} outer{outer_fold_index} final",
    )

    test_accuracy = calculate_accuracy(y_outer_test, y_test_pred)
    test_macro_f1 = calculate_macro_f1(y_outer_test, y_test_pred)
    test_weighted_f1 = calculate_weighted_f1(y_outer_test, y_test_pred)
    per_class_f1 = calculate_per_class_f1(y_outer_test, y_test_pred)

    print(f">> Outer fold {outer_fold_index} TEST:  acc={test_accuracy:.4f}  macroF1={test_macro_f1:.4f}  weightedF1={test_weighted_f1:.4f}")


    fold_row = {
        "context_level": CONTEXT_COLUMN,
        "representation": "deberta_base_finetune_bs32_fp32",
        "outer_fold": outer_fold_index,
        "best_lr": best_lr,
        "best_inner_macro_f1": best_inner_macro_f1,
        "test_accuracy": test_accuracy,
        "test_macro_f1": test_macro_f1,
        "test_weighted_f1": test_weighted_f1,
        "lr_scores_json": json.dumps({f"{k:.0e}": v for k, v in all_lr_scores.items()}),
    }
    fold_df = pd.DataFrame([fold_row])
    write_header = not os.path.exists(PROGRESS_PATH)
    fold_df.to_csv(PROGRESS_PATH, mode="a", header=write_header, index=False)

    per_class_row = {"context_level": CONTEXT_COLUMN, "outer_fold": outer_fold_index}
    for class_id, f1_value in per_class_f1.items():
        per_class_row[f"class_{class_id}_f1"] = f1_value
    pc_df = pd.DataFrame([per_class_row])
    write_pc_header = not os.path.exists(PER_CLASS_PATH)
    pc_df.to_csv(PER_CLASS_PATH, mode="a", header=write_pc_header, index=False)

    completed_folds.add(outer_fold_index)
    print(f">> Saved progress: {PROGRESS_PATH}")
    print(f">> Saved per-class F1: {PER_CLASS_PATH}")

print("\n" + "="*60)
print("ALL AVAILABLE OUTER FOLDS COMPLETE")
print("="*60)

Outer fold count: 10
Number of classes: 10
Progress path: /content/drive/MyDrive/Colab Notebooks/SML/deberta_L4_stable_deberta_base_bs32_fp32_lr2e5_5e5_8e5_fold_progress.csv

Fresh start. No prior progress file found.

>> Context L4 | Outer fold 0 | train=18000 test=2000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


      Train call L4/outer9 inner_lr=8e-05_fold=2 | n_train=12000 n_eval=6000 lr=8e-05 epochs=3 batch=32 bf16=False fp16=False
      epoch=1/3 mean_loss=1.0161 last_loss=0.6919
      epoch=2/3 mean_loss=0.4983 last_loss=0.3246
      epoch=3/3 mean_loss=0.2646 last_loss=0.2335
      Prediction distribution L4/outer9 inner_lr=8e-05_fold=2: {0: 644, 1: 605, 2: 633, 3: 608, 4: 633, 5: 576, 6: 619, 7: 558, 8: 547, 9: 577}
    [Inner] L4/outer9 lr=8e-05  fold=2  macroF1=0.8344
  [Inner] L4/outer9 lr=8e-05  avg macroF1=0.8223
>> Best lr for outer fold 9: 5e-05  (inner macroF1=0.8227)


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


      Train call L4 outer9 final | n_train=18000 n_eval=2000 lr=5e-05 epochs=3 batch=32 bf16=False fp16=False
      epoch=1/3 mean_loss=0.9302 last_loss=0.5393
      epoch=2/3 mean_loss=0.4343 last_loss=0.7285
      epoch=3/3 mean_loss=0.2365 last_loss=0.1274
      Prediction distribution L4 outer9 final: {0: 188, 1: 195, 2: 211, 3: 184, 4: 193, 5: 204, 6: 198, 7: 189, 8: 211, 9: 227}
>> Outer fold 9 TEST:  acc=0.8320  macroF1=0.8329  weightedF1=0.8329
>> Saved progress: /content/drive/MyDrive/Colab Notebooks/SML/deberta_L4_stable_deberta_base_bs32_fp32_lr2e5_5e5_8e5_fold_progress.csv
>> Saved per-class F1: /content/drive/MyDrive/Colab Notebooks/SML/deberta_L4_stable_deberta_base_bs32_fp32_lr2e5_5e5_8e5_per_class_f1.csv

ALL AVAILABLE OUTER FOLDS COMPLETE


## Summary

In [ ]:

if not os.path.exists(PROGRESS_PATH):
    raise FileNotFoundError(f"No progress file found: {PROGRESS_PATH}")

fold_df = pd.read_csv(PROGRESS_PATH)
fold_df = fold_df.drop_duplicates(subset=["outer_fold"], keep="last")
completed = fold_df["outer_fold"].nunique()
print(f"Loaded {completed} unique outer-fold rows from {PROGRESS_PATH}")

assert completed == OUTER_FOLDS, (
    f"Only {completed} folds completed, expected {OUTER_FOLDS}. "
    "Do not report this summary until all outer folds are complete."
)

summary = {
    "context_level": CONTEXT_COLUMN,
    "representation": "deberta_base_finetune_bs32_fp32",
    "test_accuracy_mean":    fold_df["test_accuracy"].mean(),
    "test_accuracy_std":     fold_df["test_accuracy"].std(),
    "test_macro_f1_mean":    fold_df["test_macro_f1"].mean(),
    "test_macro_f1_std":     fold_df["test_macro_f1"].std(),
    "test_weighted_f1_mean": fold_df["test_weighted_f1"].mean(),
    "test_weighted_f1_std":  fold_df["test_weighted_f1"].std(),
    "best_lr_mode":          fold_df["best_lr"].mode().iloc[0],
    "best_lr_counts":        json.dumps(fold_df["best_lr"].value_counts().to_dict()),
}
summary_df = pd.DataFrame([summary])
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f"\nSaved summary to {SUMMARY_PATH}")
summary_df

Loaded 10 unique outer-fold rows from /content/drive/MyDrive/Colab Notebooks/SML/deberta_L4_stable_deberta_base_bs32_fp32_lr2e5_5e5_8e5_fold_progress.csv

Saved summary to /content/drive/MyDrive/Colab Notebooks/SML/deberta_L4_stable_deberta_base_bs32_fp32_lr2e5_5e5_8e5_summary.csv


,context_level,representation,test_accuracy_mean,test_accuracy_std,test_macro_f1_mean,test_macro_f1_std,test_weighted_f1_mean,test_weighted_f1_std,best_lr_mode,best_lr_counts
0,L4,deberta_base_finetune_bs32_fp32,0.83575,0.006542,0.835378,0.006606,0.835378,0.006606,0.00005,"{""5e-05"": 9, ""2e-05"": 1}"
